## Model Training — Nigerian Credit Risk Pipeline

### Objective
Train and tune three classification models to predict loan default probability.
Models are evaluated using ROC-AUC, the industry standard metric for 
probability of default (PD) models in credit risk.

### Approach
- Baseline models trained first to establish reference performance
- Hyperparameter tuning via GridSearchCV with 5-fold cross validation
- Class imbalance handled via class_weight='balanced' (LR, RF) and 
  scale_pos_weight (XGBoost)
- Champion model selected based on test set AUC

In [ ]:
# import the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score
import warnings
from sklearn.exceptions import DataConversionWarning
import joblib


In [2]:
warnings.filterwarnings("ignore", category=UserWarning, 
                        module="sklearn.preprocessing._encoders")

In [3]:
    # Model registry
MODEL_REGISTRY = {
    "LogisticRegression": LogisticRegression(max_iter=2000, class_weight='balanced'),
    "RandomForest": RandomForestClassifier(random_state=42, class_weight='balanced'),
    "xgboost": xgb.XGBClassifier(random_state=42, scale_pos_weight=4)
    }

In [4]:
def build_pipeline(model_name, numerical, categorical):

    categorical_preprocessor = Pipeline(steps=[
        ("cat", OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
    ])

    numerical_preprocessor = Pipeline(steps=[
        ("numeric", SimpleImputer(strategy='median')),
        ("scaler", StandardScaler())
    ]) 

    preprocessor = ColumnTransformer(transformers=[
        ('cat', categorical_preprocessor, categorical),
        ('num', numerical_preprocessor, numerical)
    ])

    # Full ML pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model_name)
    ])
    
    return pipeline



In [5]:
def train_base_model(model_name, X_train, X_test, y_train, y_test, numerical, categorical):

    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model: {model_name}. Choose: {list(MODEL_REGISTRY.keys())}")
    
    # Build pipeline
    pipeline = build_pipeline(MODEL_REGISTRY[model_name], numerical, categorical)

    # fit dataset to model
    model = pipeline.fit(X_train, y_train)

    # predict
    y_pred = model.predict_proba(X_test)[:, 1]

    # evaluate
    auc_score = roc_auc_score(y_test, y_pred)

    return {
    'model_name': model_name,
    'auc_score': auc_score,
    'fitted_pipeline': model
    }



    

In [6]:
def tune_model(model_name, X_train, y_train, numerical, categorical, params=None, cv=5, scoring="roc_auc"):
    
    # Default params for each model
    DEFAULT_PARAMS = {
        "LogisticRegression": {'model__C': [0.1, 0.3, 1.0]},
        "RandomForest": {'model__n_estimators': [100, 200], 'model__max_depth': [6, 10]},
        "xgboost": {'model__n_estimators': [200, 500], 'model__max_depth': [4, 6]}
    }
    
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model: {model_name}. Choose: {list(MODEL_REGISTRY.keys())}")
    
    # Build & tune
    pipeline = build_pipeline(MODEL_REGISTRY[model_name], numerical, categorical)
    parameters = params or DEFAULT_PARAMS[model_name]
    
    print(f"Tuning {model_name}...")
    grid = GridSearchCV(pipeline, param_grid=parameters, cv=cv, scoring=scoring, n_jobs=-1)
    grid.fit(X_train, y_train)
    
    return {
        'best_estimator': grid.best_estimator_,
        'best_score': grid.best_score_, 
        'best_parameters': grid.best_params_,
        'cv_results': grid.cv_results_
    }

In [7]:
# load the dataset
path = '../data/features/loan_features.csv' 
df = pd.read_csv(path)

numerical = list(df.select_dtypes(include=['int64', 'float64']).columns)
if 'default_flag' in numerical:
    numerical.remove('default_flag')
        
categorical = list(df.select_dtypes(include=['object', 'str']).columns)

X = df[numerical+categorical]
y = df['default_flag']

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y)


# tuned = tune_model("LogisticRegression", X_train, y_train, numerical, categorical,
#           params={'model__C': [0.1, 0.3, 0.5, 1]}, cv=5, scoring="roc_auc")

train_log = train_base_model("LogisticRegression", X_train, X_test, y_train, y_test, numerical, categorical,
                             )
train_rf = train_base_model("RandomForest", X_train, X_test, y_train, y_test, numerical, categorical,
                             )
train_xgb = train_base_model("xgboost", X_train, X_test, y_train, y_test, numerical, categorical,
                             )

In [8]:
print(f'lr_auc: {train_log['auc_score']}, rf_auc: {train_rf['auc_score']}, xgb_auc: {train_xgb['auc_score']}')

lr_auc: 0.7165770999068155, rf_auc: 0.7052479547053022, xgb_auc: 0.6949268720486432


### Baseline Results
Logistic Regression outperforms tree-based models at baseline, suggesting
partially linear relationships in the data. XGBoost underperforms at baseline
as expected — it requires tuning to perform well.

In [9]:
lr_tuned = tune_model("LogisticRegression", X_train, y_train, numerical, categorical,
                    cv=5, scoring="roc_auc")

rf_tuned = tune_model("RandomForest", X_train, y_train, numerical, categorical,
                    cv=5, scoring="roc_auc")

xgb_tuned = tune_model("xgboost", X_train, y_train, numerical, categorical,
                    cv=5, scoring="roc_auc")

xgb_tuned_v2 = tune_model("xgboost", X_train, y_train, numerical, categorical,
    params={
        'model__n_estimators': [300, 500],
        'model__max_depth': [3, 4, 6],
        'model__learning_rate': [0.01, 0.05, 0.1],
        'model__min_child_weight': [1, 5, 10],
        'model__subsample': [0.8, 1.0]
    },
    cv=5, scoring="roc_auc")

Tuning LogisticRegression...
Tuning RandomForest...
Tuning xgboost...
Tuning xgboost...


### Hyperparameter Tuning
GridSearchCV with 5-fold stratified cross validation. XGBoost required a 
second deeper tuning round with learning_rate, min_child_weight and subsample 
parameters to reach its performance ceiling.

In [11]:
y_pred_lr = lr_tuned['best_estimator'].predict_proba(X_test)[:,1]
y_pred_rf = rf_tuned['best_estimator'].predict_proba(X_test)[:,1]
y_pred_xgb = xgb_tuned['best_estimator'].predict_proba(X_test)[:,1]
y_pred_xgb_v2 = xgb_tuned_v2['best_estimator'].predict_proba(X_test)[:,1]

lr_tuned_auc = roc_auc_score(y_test, y_pred_lr)
rf_tuned_auc = roc_auc_score(y_test, y_pred_rf)
xgb_tuned_auc = roc_auc_score(y_test, y_pred_xgb)
xgbv2_tuned_auc = roc_auc_score(y_test, y_pred_xgb_v2)

print(f'lr_tuned_auc: {lr_tuned_auc}, rf_tuned_auc: {rf_tuned_auc}, xgb_tuned_auc: {xgb_tuned_auc}, xgbv2_tuned_auc: {xgbv2_tuned_auc}')

lr_tuned_auc: 0.7170662553775209, rf_tuned_auc: 0.7142054920435644, xgb_tuned_auc: 0.7004954270528397, xgbv2_tuned_auc: 0.7223039110927107


In [12]:
print(xgb_tuned_v2['best_parameters'])

{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__min_child_weight': 10, 'model__n_estimators': 500, 'model__subsample': 0.8}


### Model Selection Summary
| Model | Baseline AUC | Tuned AUC |
|---|---|---|
| Logistic Regression | 0.7166 | 0.7171 |
| Random Forest | 0.7052 | 0.7142 |
| XGBoost v1 | 0.6949 | 0.7005 |
| XGBoost v2 | - | 0.7223 |

**Champion model: XGBoost v2 (AUC: 0.7223)**
- Best parameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__min_child_weight': 10, 'model__n_estimators': 500, 'model__subsample': 0.8}
- Saved to models/best_model_xgb.pkl for deployment.

### Next Step
Proceeding to 05_model_evaluation.ipynb for deep evaluation including
ROC curve, confusion matrix at multiple thresholds, 
feature importance and business impact analysis.

In [ ]:
joblib.dump(xgb_tuned_v2['best_estimator'], '../models/best_model_xgb.pkl')

['../models/best_model_xgb.pkl']